# AI Workforce Statements: Complete Revelio Firm Matching

## Part 0 Notebook Setup and Research Scope

**Purpose.** Extend the **provisional** historical firm crosswalk prepared in
`01_build_firm_crosswalk.ipynb` by resolving remaining Revelio company identities
for firms represented in the S&P 500 from **January 1, 2018 through December 31, 2025**.

This notebook is a follow-up to Notebook 01, **not a replacement** for its CRSP–CCM
historical linking and Compustat candidate universe. The last saved baseline reported:

- **652** candidate GVKEY firms.
- **539** firms with direct Revelio GVKEY candidates.
- **26** additional firms with local CIK and/or full-CUSIP candidates.
- **87** firms without any candidate in the *ticker-filtered* Revelio extract.

These counts describe **candidate availability**, not 565 accepted entity matches.
The exported Notebook 01 crosswalk is provisional; RCIDs require explicit review.

### Sources and units of observation

| Source | Unit of observation | Relevant identifiers |
|---|---|---|
| Notebook 01 `crosswalk.parquet` | One provisional GVKEY firm | `firm_id`, GVKEY, candidate identity/status |
| Notebook 01 `crosswalk_membership_bridge.parquet` | Historical membership interval | PERMNO, PERMCO, dated CCM evidence |
| Compustat Fundamentals Annual | Annual company observation | GVKEY, CIK, CUSIP, ticker, company name |
| Existing Revelio mapping | Ticker-filtered company candidates | RCID, CIK, CUSIP, ticker, name, LinkedIn URL |
| **Full Revelio company list (to obtain)** | Revelio company record | RCID and available identifiers / hierarchy |

**Matching order.** Search the *full* Revelio company list using **CIK → CUSIP → ticker
→ exact normalized company name → LinkedIn URL → manual review**. Keep all candidate
identities and their evidence. An exact identifier match is **not** automatic approval:
historical validity, renames, parent/subsidiary scope, and one-to-many cases require review.

**Identifier integrity.** Keep GVKEY, CIK, CUSIP, ticker, and RCID as strings where
appropriate. Preserve leading zeros. Distinguish issue-level CUSIP from company identity.
Never overwrite original source identifiers with normalized comparison keys.

**Data handling.** Licensed company-level data and review tables remain local under
`data/`. Preserve the Notebook 01 outputs. No original source file or v1 crosswalk
is modified. Review cell outputs before sharing this notebook publicly.

**Current status.** Part 0 provides setup and an input inventory only. No matching,
acceptance, or export is performed in this starter notebook. Subsequent Parts require
researcher review before implementation.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

study_start = pd.Timestamp("2018-01-01")
study_end = pd.Timestamp("2025-12-31")

# Same project-root convention used by Notebook 01.
for project_root in [Path("."), Path(".."), Path("../..")]:
    if (project_root / "data" / "raw").is_dir():
        break
else:
    raise FileNotFoundError("Run from the project root, code/, or code/notebooks/.")

print(f"Study period: {study_start.date()} to {study_end.date()}")
print(f"Project root: {project_root.resolve()}")
print(f"pandas version: {pd.__version__}")

## Part 1 Input Inventory and Baseline Verification

### 1.1 Required inputs and availability

Check the existing v1 outputs and reference files **without loading or modifying** them.
The filename and location of the full Revelio company list are intentionally left
unassigned until the new extract is available. Do not mistake the old ticker-filtered
mapping for the full company universe.

In [ ]:
input_paths = {
    "crosswalk_v1": project_root / "data/clean/crosswalk.parquet",
    "membership_bridge_v1": project_root / "data/clean/crosswalk_membership_bridge.parquet",
    "compustat": project_root / "data/raw/compustat/Compustat Fundamentals Annual.csv",
    "revelio_ticker_filtered": project_root / "data/raw/revelio/revelio_company_mapping_20261023.csv",
}

# Set only after obtaining and inspecting the FULL company-reference export.
revelio_full_company_path = None

inventory = pd.DataFrame([
    {
        "input": name,
        "relative_path": path.relative_to(project_root).as_posix(),
        "exists": path.is_file(),
        "size_mb": round(path.stat().st_size / 1_000_000, 2) if path.is_file() else pd.NA,
    }
    for name, path in input_paths.items()
])
display(inventory)

print("Full Revelio company reference:",
      "NOT CONFIGURED" if revelio_full_company_path is None else revelio_full_company_path)
# Inventory only: no dataframes loaded, no identities selected, no files written.

### 1.2 Baseline checks

Read the provisional crosswalk, Compustat, and the **existing ticker-filtered** Revelio
extract only. All reconstructed evidence stays in memory; no source or crosswalk export
is written. The saved GVKEY universe is fixed for this audit.

#### 1.2.1 Saved baseline and identifier completeness

Retain the loaded tables unchanged. CSV identifiers use string dtype with only empty
fields treated as missing (`NA` can be a ticker). Comparison keys are separate columns.

In [ ]:
import json

crosswalk_v1 = pd.read_parquet(input_paths["crosswalk_v1"])
compustat_source = pd.read_csv(
    input_paths["compustat"], dtype="string", keep_default_na=False, na_values=[""],
)
revelio_local_source = pd.read_csv(
    input_paths["revelio_ticker_filtered"], dtype="string", keep_default_na=False, na_values=[""],
)
assert len(crosswalk_v1) == 652, "Saved firm baseline changed; investigate before proceeding."
assert crosswalk_v1["gvkey"].notna().all() and crosswalk_v1["gvkey"].is_unique
assert crosswalk_v1["firm_id"].is_unique
assert crosswalk_v1["firm_id"].eq("gvkey:" + crosswalk_v1["gvkey"]).all()
assert crosswalk_v1["gvkey"].str.fullmatch(r"[0-9]{6}").all()

completeness_rows = []
for column in ["gvkey", "conm", "company_name", "cik", "cusip", "cusip8", "ticker", "rcid"]:
    exists = column in crosswalk_v1
    present = (crosswalk_v1[column].astype("string").str.strip().replace("", pd.NA).notna().sum()
               if exists else 0)
    completeness_rows.append({"field": column, "in_schema": exists,
                              "present_firms": int(present), "missing_firms": 652 - int(present)})
display(pd.DataFrame(completeness_rows))
display(crosswalk_v1.groupby(["rcid_status", "method_category"], dropna=False)
        .size().rename("firms").to_frame())
print("Accepted RCIDs:", crosswalk_v1["rcid"].notna().sum())
assert crosswalk_v1["rcid"].isna().all(), "Saved acceptance state changed."
assert crosswalk_v1["decision_basis"].isna().all()

#### 1.2.2 Reconstruct Compustat evidence and diagnose missing scalars

Notebook 01 Part 4 uses **all downloaded annual rows** for the candidate GVKEYs;
no year, format, or membership-date filter is applied to identifier evidence. Keep
source-row numbers, dates, formats, raw identifiers, and every distinct observed value.
Do not choose a latest or preferred annual observation.

Notebook 01 Part 7.2 aggregates tuples, then accepts a scalar only when
`isinstance(values, tuple)` and its length is one. The probe below tests that exact
aggregation on this pandas version. A list returned by aggregation fails this guard,
even when it contains exactly one identifier. Company name and full CUSIP were not
included in the final schema; `cusip8` was a derived prefix, not the original CUSIP.

In [ ]:
compustat_evidence = compustat_source.copy()
compustat_evidence.columns = compustat_evidence.columns.str.lower()
compustat_evidence["compustat_row"] = range(1, len(compustat_evidence) + 1)
compustat_evidence["gvkey_key"] = compustat_evidence["gvkey"].str.strip().replace("", pd.NA)
compustat_evidence = compustat_evidence.loc[
    compustat_evidence["gvkey_key"].isin(crosswalk_v1["gvkey"])
].copy()
for column in ["conm", "cik", "cusip", "tic"]:
    compustat_evidence[column + "_key"] = compustat_evidence[column].str.strip().replace("", pd.NA)
# Notebook 01 Part 5: positive digits-only CIK, with zeros stripped ONLY in comparison keys.
cik_digits = compustat_evidence["cik_key"].str.fullmatch(r"[0-9]+").fillna(False)
cik_comparison = compustat_evidence["cik_key"].str.lstrip("0").replace("", "0")
compustat_evidence["cik_for_comparison"] = cik_comparison.where(cik_digits.astype(bool) & cik_comparison.ne("0").fillna(False).astype(bool))
cusip_comparison = compustat_evidence["cusip_key"].str.upper()
compustat_evidence["cusip_for_comparison"] = cusip_comparison.where(
    cusip_comparison.str.fullmatch(r"[0-9A-Z*@#]{8,9}").fillna(False)
)
compustat_evidence["cusip8_observed"] = compustat_evidence["cusip_for_comparison"].str[:8]

# Explicit records keep tuple containers independent of pandas GroupBy dtype coercion.
evidence_rows = []
for gvkey, annual in compustat_evidence.groupby("gvkey_key", sort=True):
    record = {"gvkey": gvkey, "annual_rows": len(annual)}
    for column in ["conm", "cik", "cusip", "tic", "cik_for_comparison", "cusip8_observed"]:
        values = annual[column]
        present = values.notna() & values.str.strip().ne("").fillna(False)
        record[column + "_values"] = tuple(sorted(values.loc[present].unique()))
    evidence_rows.append(record)
firm_evidence = crosswalk_v1[["gvkey"]].merge(
    pd.DataFrame(evidence_rows), on="gvkey", how="left", validate="one_to_one",
)
for column in firm_evidence.columns[2:]:
    firm_evidence[column] = [value if isinstance(value, tuple) else () for value in firm_evidence[column]]
firm_evidence["annual_rows"] = firm_evidence["annual_rows"].fillna(0).astype("int64")

evidence_completeness = pd.DataFrame([
    {"evidence": column, "firms_with_value": int(firm_evidence[column].map(len).gt(0).sum()),
     "firms_missing": int(firm_evidence[column].map(len).eq(0).sum()),
     "firms_multiple_values": int(firm_evidence[column].map(len).gt(1).sum())}
    for column in ["conm_values", "cik_values", "cusip_values", "tic_values"]
])
display(evidence_completeness)
print("Compustat annual rows:", len(compustat_evidence))
print("Firms without Compustat rows:", firm_evidence["annual_rows"].eq(0).sum())

scalar_diagnosis = []
for source, target in [("cik_for_comparison", "cik"), ("cusip8_observed", "cusip8"), ("tic_key", "ticker")]:
    probe = compustat_evidence.groupby("gvkey_key").agg(
        values=(source, lambda values: tuple(sorted(values.dropna().unique())))
    )["values"]
    scalar_diagnosis.append({
        "saved_field": target,
        "aggregation_container_types": ", ".join(sorted({type(value).__name__ for value in probe})),
        "single_value_groups": int(probe.map(len).eq(1).sum()),
        "pass_original_tuple_guard": int(probe.map(lambda value: isinstance(value, tuple) and len(value) == 1).sum()),
        "saved_nonnull": int(crosswalk_v1[target].notna().sum()),
    })
display(pd.DataFrame(scalar_diagnosis))
# Raw strings retain source padding; these counts do not normalize or replace them.
display(pd.Series({column: int(compustat_evidence[column].str.startswith("0").fillna(False).sum())
                   for column in ["gvkey", "cik", "cusip"]}, name="source_rows_with_leading_zero").to_frame())

#### 1.2.3 Reproduce original local candidate definitions

Use Notebook 01 Parts 5?6: a nonblank RCID with an exact trimmed GVKEY, a positive
digits-only CIK equal after zero normalization, or an exact **full** valid 8/9-character
CUSIP after uppercasing. Never match CUSIP prefixes, tickers, or names in this baseline.
All observed Compustat identifiers participate, and conflicting Revelio GVKEYs do not
exclude supplementary candidates. Direct GVKEY takes precedence in the category only.

Compare every reconstructed `(GVKEY, RCID, method)` against the saved candidate JSON,
not merely the aggregate counts. No candidate is accepted.

In [ ]:
revelio_local = revelio_local_source.copy()
revelio_local.columns = revelio_local.columns.str.lower()
for column in ["gvkey", "rcid", "cik", "cusip"]:
    revelio_local[column + "_key"] = revelio_local[column].str.strip().replace("", pd.NA)
r_cik = revelio_local["cik_key"].str.lstrip("0").replace("", "0")
revelio_local["cik_for_comparison"] = r_cik.where(
    revelio_local["cik_key"].str.fullmatch(r"[0-9]+").fillna(False).astype(bool)
    & r_cik.ne("0").fillna(False).astype(bool)
)
r_cusip = revelio_local["cusip_key"].str.upper()
revelio_local["cusip_for_comparison"] = r_cusip.where(
    r_cusip.str.fullmatch(r"[0-9A-Z*@#]{8,9}").fillna(False)
)
local_with_rcid = revelio_local.loc[revelio_local["rcid_key"].notna()]
direct = local_with_rcid.loc[local_with_rcid["gvkey_key"].isin(crosswalk_v1["gvkey"])]
reconstructed_candidates = {(row.gvkey_key, row.rcid_key, "direct_gvkey")
                            for row in direct.itertuples(index=False)}
for key, method in [("cik_for_comparison", "cik"), ("cusip_for_comparison", "full_cusip")]:
    left = compustat_evidence.loc[compustat_evidence[key].notna(), ["gvkey_key", key]].drop_duplicates()
    right = local_with_rcid.loc[local_with_rcid[key].notna(), ["rcid_key", key]].drop_duplicates()
    matched = left.merge(right, on=key, how="inner")
    reconstructed_candidates.update((row.gvkey_key, row.rcid_key, method)
                                    for row in matched.itertuples(index=False))

saved_candidates = set()
for row in crosswalk_v1.itertuples(index=False):
    for candidate in json.loads(row.revelio_candidates_json):
        saved_candidates.update((row.gvkey, candidate["rcid"], method) for method in candidate["methods"])
print("Reconstructed / saved candidate-method triples:", len(reconstructed_candidates), len(saved_candidates))
print("Added / missing triples:", len(reconstructed_candidates - saved_candidates),
      len(saved_candidates - reconstructed_candidates))
assert reconstructed_candidates == saved_candidates, "Local evidence differs from v1; investigate."

candidate_status = crosswalk_v1[["gvkey"]].copy()
for method in ["direct_gvkey", "cik", "full_cusip"]:
    keys = {gvkey for gvkey, rcid, evidence_method in reconstructed_candidates if evidence_method == method}
    candidate_status[method] = candidate_status["gvkey"].isin(keys)
candidate_status["candidate_status"] = "No local candidate"
no_direct = ~candidate_status["direct_gvkey"]
for cik_flag, cusip_flag, label in [
    (True, False, "Additional CIK only"),
    (False, True, "Additional full CUSIP only"),
    (True, True, "Additional CIK and full CUSIP"),
]:
    mask = no_direct & candidate_status["cik"].eq(cik_flag) & candidate_status["full_cusip"].eq(cusip_flag)
    candidate_status.loc[mask, "candidate_status"] = label
candidate_status.loc[candidate_status["direct_gvkey"], "candidate_status"] = "Direct GVKEY candidate"
assert candidate_status["candidate_status"].eq(crosswalk_v1["method_category"]).all()
no_local = candidate_status["candidate_status"].eq("No local candidate")
assert no_local.eq(crosswalk_v1["no_local_candidate"]).all()
assert no_local.eq(crosswalk_v1["rcid_status"].eq("no_local_candidate")).all()
assert int(candidate_status["direct_gvkey"].sum()) == 539
assert int((no_direct & ~no_local).sum()) == 26
assert int(no_local.sum()) == 87
display(candidate_status.groupby("candidate_status").size().rename("firms").to_frame())
assert len(saved_candidates) == len(reconstructed_candidates) == 1638
print("Verified all 1,638 saved candidate-method triples; no candidate accepted.")


#### 1.2.4 Review table: firms without local candidates

`baseline_review` contains all 652 firms; `no_local_candidate_review` displays all 87
without local candidates. Identifier columns are tuples of **original Compustat values**,
including leading zeros; multiple values remain visible and `()` means no source evidence.
These are observed reference values, not verified historical identifiers. Source-row,
report-date, and format details remain available in `compustat_evidence`.

In [ ]:
baseline_review = firm_evidence.merge(
    candidate_status[["gvkey", "candidate_status"]], on="gvkey", validate="one_to_one",
).rename(columns={"conm_values": "company_name", "cik_values": "cik",
                  "cusip_values": "cusip", "tic_values": "ticker"})
review_columns = ["gvkey", "company_name", "cik", "cusip", "ticker", "candidate_status"]
no_local_candidate_review = baseline_review.loc[
    baseline_review["candidate_status"].eq("No local candidate"), review_columns
].sort_values("gvkey").reset_index(drop=True)
assert len(no_local_candidate_review) == 87
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
    display(no_local_candidate_review)
# Reload only to verify that the in-memory audit did not alter the saved crosswalk.
pd.testing.assert_frame_equal(crosswalk_v1, pd.read_parquet(input_paths["crosswalk_v1"]))

### Findings

- The saved baseline has **652 unique GVKEY firms**, **539 direct candidates**, **26
  supplementary candidates** (25 CIK plus full CUSIP, one full CUSIP only), and **87
  without local candidates**. All accepted RCIDs and decision bases are null.
- The exported CIK, CUSIP8, and ticker scalars are entirely null. Notebook 01 Part 7.2
  uses a tuple-only guard; with pandas 3.0.1, string-column aggregation returns lists,
  so even singleton evidence is discarded. The diagnostic reproduces this failure
  against the current source. The historical export environment is not independently
  established, so this is a reproduced mechanism consistent with the saved artifact,
  not proof of its exact execution history.
- Company name and original full CUSIP were omitted from Notebook 01's export schema.
  Its CIK scalar was intended to be zero-normalized and CUSIP8 truncated; neither is
  a substitute for original source identifiers. This audit retains raw values and
  reconstructs evidence separately, without patching the v1 artifact.
- The same tuple-only check appears in Notebook 01 Part 5's `company_evidence` handling.
  It can empty the displayed observed-name/identifier evidence and distort identity
  review flags. Direct GVKEY availability and supplementary identifier joins use
  separate paths. Exact equality of saved and reconstructed candidate-method triples
  tests candidate preservation; it does **not** validate the saved identity-review flags.
- The 87 firms are absent only under the original three methods in the **ticker-filtered
  local extract**. This does not establish absence from the full Revelio universe.
  Ticker/name searches, historical validity, hierarchy conflicts, annual-format choices,
  and RCID adjudication remain unresolved. Part 2 is not implemented.
- Completeness and multiplicity diagnostics above report the actual source gaps without
  selecting preferred records. All reconstructed evidence remains in memory; no source,
  crosswalk, membership bridge, schema, or log is exported or overwritten.

## Part 2 Full Revelio Company Reference and Multi-method Matching

### 2.1 Full Revelio Company Reference

This section inventories and validates the full company reference only. Part 2.2 matching
is not implemented. No RCID is accepted and no research data are written.

#### 2.1.1 Local availability and provenance

On 2026-11-05, the project inventory contained only
`data/raw/revelio/revelio_company_mapping_20261023.csv`, the ticker-filtered extract.
No full company reference was found among project data files. This search does not cover
other computers or external storage. The intended destination is now fixed as
`data/raw/revelio/revelio_company_reference_full.csv`; specifying it does not download it.
The existing Part 1.1 inventory records the earlier, unconfigured state.

The code below checks this destination on every run and never falls back to the filtered
extract. A local filename or a larger row count alone cannot prove an unrestricted export.
Record the WRDS table, retrieval date, query, and source row count after manual download.


**Download inspection update (2026-11-06).** The actual file is
`data/raw/revelio/eudkcvtewjnwtygx.csv.gz` (7,634,183,551 compressed bytes).
The path below supersedes the intended destination above. Prior findings and saved
outputs are retained as historical records, not results for this download.

A read-only gzip header inspection found 34 columns, including `rcid`, `company`,
`gvkey`, `cik`, `cusip`, `ticker`, `linkedin_url`, `ultimate_parent_rcid`,
`ultimate_parent_company_name`, `child_rcid`, and `child_company`. This is consistent
with the company-reference/mapping schema expected by this notebook; it does not
verify the WRDS relation, unrestricted export, completeness, or historical coverage.

**Current step: five-row inspection only.** Run the availability cell and the new
inspection cell below, then stop for researcher review. Part 0 supplies `project_root`,
`pd`, and `display`. All preview values remain strings, preserving leading zeros,
empty fields, and literal `NA`; original column labels are retained.

**Later cells needing review (unchanged here):**
- 2.1.2 refers to the old destination and decompression; neither is needed here.
- 2.1.3 opens the header as plain text and loads the entire CSV. It needs gzip-aware
  header reading and a bounded-memory audit design, including global duplicate,
  RCID-group, and hierarchy checks across chunks.
- 2.1.4 invokes that full-file audit and hashes the entire file. Do not run it yet.
- 2.1.5 and saved outputs describe earlier preparation, not validation of this export.


In [ ]:
from datetime import datetime, timezone
import csv
import hashlib

revelio_full_company_path = project_root / "data/raw/revelio/eudkcvtewjnwtygx.csv.gz"
revelio_full_source = None
revelio_full_audit = None
revelio_full_status = "PENDING_INSPECTION" if revelio_full_company_path.is_file() else "FILE_NOT_FOUND"
print("Checked (UTC):", datetime.now(timezone.utc).isoformat())
print("Downloaded reference (coverage unverified):", revelio_full_company_path.resolve())
display(pd.DataFrame([
    {"file": path.name, "bytes": path.stat().st_size}
    for path in sorted((project_root / "data/raw/revelio").iterdir()) if path.is_file()
]))
print("Full reference exists:", revelio_full_company_path.is_file())
if not revelio_full_company_path.is_file():
    print("FILE_NOT_FOUND: check the downloaded gzip path before inspection.")

# Fill from the actual WRDS export, not filesystem timestamps or filename guesses.
revelio_full_provenance = {
    "wrds_relation": None,
    "retrieved_utc": None,
    "query_or_download_settings": None,
    "wrds_row_count": None,
    "unrestricted_export_confirmed": False,
}

In [ ]:
# 2.1.1a Inspection only: gzip header and first five rows. STOP after this cell.
import gzip
from itertools import islice

with gzip.open(revelio_full_company_path, "rt", encoding="utf-8-sig", newline="") as handle:
    revelio_preview_reader = csv.reader(handle)
    revelio_preview_columns = next(revelio_preview_reader)
    revelio_preview_rows = list(islice(revelio_preview_reader, 5))

# csv.reader returns strings: no numeric inference, normalization, or NA conversion.
revelio_full_preview = pd.DataFrame(
    revelio_preview_rows, columns=revelio_preview_columns, dtype="string",
)
revelio_compressed_bytes = revelio_full_company_path.stat().st_size
print("Compressed file:", revelio_full_company_path.resolve())
print(f"Compressed file size: {revelio_compressed_bytes:,} bytes "
      f"({revelio_compressed_bytes / 1024**3:.3f} GiB)")
print("Number of columns:", len(revelio_preview_columns))
print("Column names:")
display(revelio_preview_columns)
print("First 5 rows (or fewer if the file ends):")
with pd.option_context("display.max_columns", None):
    display(revelio_full_preview)
print("STOP for researcher review. Schema/sample and file size do not establish full coverage.")


#### 2.1.2 Manual WRDS acquisition instructions

Use **WRDS > Get Data > Revelio Labs > Company Mapping / Company Reference** under
your institution's subscription. Request the unrestricted company mapping/reference,
not profiles, positions, job postings, a sample, or the historical-ticker query aid.
The vendor documents the fields in its
[Company Reference dictionary](https://www.datasets.reveliolabs.com/data.html).
Authenticated WRDS access and the current relation name have **not** been verified here.
`revelio.company_mapping` is a candidate relation to check, not a claimed successful query.

In an authenticated WRDS PostgreSQL session, first discover the accessible relation and
its actual schema (these queries do not download company records):

```sql
SELECT table_schema, table_name
FROM information_schema.columns
WHERE table_schema ILIKE '%revelio%'
GROUP BY table_schema, table_name
HAVING bool_or(column_name = 'rcid')
   AND bool_or(column_name = 'company')
   AND bool_or(column_name = 'ultimate_parent_rcid')
ORDER BY table_schema, table_name;

-- Run this if discovery confirms revelio.company_mapping:
SELECT column_name, data_type, ordinal_position
FROM information_schema.columns
WHERE table_schema = 'revelio' AND table_name = 'company_mapping'
ORDER BY ordinal_position;
```

After confirming that this is the unrestricted company reference, the exact extraction
query for that relation is:

```sql
SELECT * FROM revelio.company_mapping;
```

Use the discovered schema/table in place of `revelio.company_mapping` if different.
Do not add `WHERE`, `LIMIT`, `DISTINCT`, ticker uploads, public-company filters,
country filters, or 2018-2025 date restrictions. Retain all columns and rows. In the
WRDS web form choose the entire universe / search all companies and all variables;
if the form requires a ticker restriction, use the authenticated SQL route instead.
Select CSV output with a header; decompress a compressed export without opening and
resaving it in Excel. Transfer the completed file to:

`data/raw/revelio/revelio_company_reference_full.csv`

Do not replace an existing file or rename/copy the ticker-filtered extract into this path.
If the destination already exists, inspect its provenance before proceeding. Obtain
`SELECT COUNT(*) FROM revelio.company_mapping;` from the same snapshot as the export
(or the WRDS job's result count), then fill `revelio_full_provenance` above and rerun
2.1.1 onward. A later count can differ if the vendor refreshed its table.
If discovery returns no table or access is denied, ask the institution's WRDS administrator
for Revelio company-reference entitlement and the correct relation; no full download
has been performed by this notebook.

Expect RCID and company name plus available CIK, CUSIP, ticker, LinkedIn URL, GVKEY,
FactSet ID, ISIN, LEI, and hierarchy fields. Missing columns are reported, not fabricated.
The vendor dictionary uses `ultimate_parent_rcid_name`; the existing local extract uses
`ultimate_parent_company_name`. Both spellings are recognized without renaming source fields.

#### 2.1.3 Part 2.1B: gzip-aware, bounded-memory diagnostics

This first pass validates the CSV header, counts parsed data rows, and summarizes
missingness and nonblank coverage for `rcid`, `company`, `gvkey`, `cik`, `cusip`,
`ticker`, and `ultimate_parent_rcid`. It replaces the former full-file loader.
The saved UnicodeDecodeError came from decoding compressed bytes as plain UTF-8;
header inspection now uses `gzip.open`, and the chunk reader explicitly uses gzip.

Header labels must be nonempty and unique after trimming and lowercasing for lookup
only. Original labels and field values are unchanged. Missing requested columns are
reported as absent, with counts and percentages unavailable rather than fabricated.
A missing RCID or company column requires review, but does not prevent counting rows.

The initial chunk size is **10,000 rows**, with only available requested columns
materialized. If none exist, one source column is read solely to count rows. Each
chunk and its temporary Series are discarded after updating scalar counters; no
row-level table, list of chunks, or identifier universe is retained or returned.
Memory depends on chunk size and field lengths, not total row count; this is not a
fixed byte limit. Long text columns outside the requested fields are not retained.
The scan still decompresses the complete file and may take time.

All loaded values use string dtype with NA inference disabled, preserving empty
strings, leading zeros, whitespace, and literal `NA`. Empty strings and whitespace-only
strings are counted separately. Missing/blank is their sum; coverage is nonblank rows
divided by all parsed data rows. Literal `NA`, `.`, and other nonblank tokens count as
present in this pass, without asserting that they are valid identifiers. A zero-row
file has undefined percentages. Blank lines are not silently skipped; this counts
CSV records, not physical lines, and quoted embedded newlines remain within records.
Parser errors abort the scan rather than silently skipping malformed records.

**Original audit objectives retained, but pending:** identifier format validity and
possible missing-token review; missingness outside these seven fields; exact duplicates;
raw and trimmed RCID uniqueness/multiplicity; per-RCID conflicting values; parent/child
missing targets, self-links, multiple targets, onward-parent joins, and name consistency.
No global duplicate/groupby/hierarchy checks, SHA256, matching, or acceptance occur here.
Header validation and nonblank coverage do not establish the WRDS relation, unrestricted
export, unique-company count, historical validity, or complete coverage.


In [ ]:
import csv
import gzip


def audit_revelio_full_reference(path, chunksize=10_000):
    """Part 2.1B only: return aggregate counts, never row-level records."""
    if isinstance(chunksize, bool) or not isinstance(chunksize, int) or chunksize <= 0:
        raise ValueError("chunksize must be a positive integer.")
    with gzip.open(path, "rt", encoding="utf-8-sig", newline="") as handle:
        header = next(csv.reader(handle, strict=True), [])
    labels = [label.strip().lower() for label in header]
    if not labels or any(not label for label in labels) or len(labels) != len(set(labels)):
        raise ValueError("Empty or ambiguous column labels; inspect the original header without rewriting it.")
    lookup = dict(zip(labels, header))
    fields = ["rcid", "company", "gvkey", "cik", "cusip", "ticker", "ultimate_parent_rcid"]
    present = [field for field in fields if field in lookup]
    selected = [lookup[field] for field in present] or [header[0]]
    counts = {field: {"empty_rows": 0, "whitespace_only_rows": 0} for field in present}
    total_rows = 0
    chunks_processed = 0

    with pd.read_csv(
        path, compression="gzip", encoding="utf-8-sig", dtype="string",
        keep_default_na=False, na_filter=False, skip_blank_lines=False,
        usecols=selected, chunksize=chunksize, on_bad_lines="error",
    ) as reader:
        for chunk in reader:
            total_rows += len(chunk)
            chunks_processed += 1
            for field in present:
                values = chunk[lookup[field]]
                empty = values.eq("")
                whitespace_only = values.str.strip().eq("") & ~empty
                counts[field]["empty_rows"] += int(empty.sum())
                counts[field]["whitespace_only_rows"] += int(whitespace_only.sum())
                del values, empty, whitespace_only
            del chunk

    records = []  # Seven aggregate records only; never source rows or chunks.
    for field in fields:
        exists = field in lookup
        empty = counts[field]["empty_rows"] if exists else pd.NA
        whitespace = counts[field]["whitespace_only_rows"] if exists else pd.NA
        missing = empty + whitespace if exists else pd.NA
        nonblank = total_rows - missing if exists else pd.NA
        records.append({
            "field": field, "column_present": exists, "total_rows": total_rows,
            "empty_rows": empty, "whitespace_only_rows": whitespace,
            "missing_or_blank_rows": missing, "nonblank_rows": nonblank,
            "missing_or_blank_pct": 100 * missing / total_rows if exists and total_rows else pd.NA,
            "nonblank_coverage_pct": 100 * nonblank / total_rows if exists and total_rows else pd.NA,
        })
    return {
        "summary": {"rows": total_rows, "columns": len(header),
                    "chunks_processed": chunks_processed, "chunk_size": chunksize},
        "columns": header,
        "missingness": pd.DataFrame(records),
    }


#### 2.1.4 Part 2.1B: run the bounded-memory scan manually

Run the updated function-definition cell above before running this cell, so the kernel
replaces the old full-file loader. This run scans the complete gzip stream in bounded
chunks and displays one seven-row summary. Only aggregate results enter
`revelio_full_audit`; `revelio_full_source` remains `None`. An interrupted or failed scan
is not marked complete and does not publish partial counts as final results. No file
is written or hashed. Stop after reviewing this pass; further diagnostics require
researcher authorization. Earlier instructions and saved outputs remain historical.

**Hierarchy interpretation retained for the future audit (not measured here):**
Self-parent references can represent top-level companies; they are not automatic errors.
Missing parent targets, multiple parents per RCID, and an ultimate parent pointing onward
to another entity require review. `child_rcid` denotes the largest subsidiary, not every
child, and can equal the company's own RCID. These fields cannot reconstruct a complete
ownership tree. See the existing [reference definitions](https://www.datasets.reveliolabs.com/data.html)
and [hierarchy methodology](https://www.datasets.reveliolabs.com/methodology.html).
An absent target can reflect coverage or export limitations, not necessarily an invalid
ID. Names can vary; no historical ownership or identity is inferred here.


In [ ]:
# Part 2.1B: rerun the function-definition cell above before this manual scan.
revelio_full_source = None
revelio_full_audit = None
revelio_full_status = "PENDING_BOUNDED_SCAN"
revelio_diagnostic_chunksize = 10_000
if not revelio_full_company_path.is_file():
    revelio_full_status = "FILE_NOT_FOUND"
    print("FILE_NOT_FOUND: check the compressed export path.")
else:
    revelio_full_status = "BOUNDED_SCAN_IN_PROGRESS"
    print(f"Scanning gzip in {revelio_diagnostic_chunksize:,}-row chunks; this may take time.", flush=True)
    try:
        revelio_full_audit = audit_revelio_full_reference(
            revelio_full_company_path, chunksize=revelio_diagnostic_chunksize,
        )
    except BaseException:
        revelio_full_status = "BOUNDED_SCAN_FAILED_OR_INTERRUPTED"
        raise
    revelio_full_status = "BOUNDED_PASS_COMPLETE_REVIEW_REQUIRED"
    print("Scan totals:", revelio_full_audit["summary"])
    with pd.option_context("display.max_columns", None):
        display(revelio_full_audit["missingness"])
    print(revelio_full_status)
    print("Coverage means nonblank rows, not valid identifiers or unique companies.")
    print("Duplicates, RCID uniqueness, hierarchy, provenance and full coverage remain unvalidated.")
    print("STOP for researcher review before additional diagnostics.")


#### 2.1.5 Findings, limitations, and next action

**Preparation finding (2026-11-05):** no full reference is available locally. The only
Revelio source is the original ticker-filtered extract; it remains unchanged. Notebook 01
and completed Part 1.2 were reviewed before adding this section. Their candidate counts
must not be presented as full-reference coverage.

**Completed:** the destination is specified, manual WRDS acquisition instructions are
provided, and read-only schema, string-integrity, missingness, RCID uniqueness, exact
record duplication, per-RCID conflicting values, and hierarchy diagnostics are prepared.
The saved execution reports `PENDING_DOWNLOAD`; it does not contain invented full-file
statistics. The validator's code checks were exercised using temporary synthetic inputs,
not a downloaded full reference.

**Still pending:** authenticated WRDS relation/entitlement confirmation, unrestricted
export, provenance and source-count documentation, then execution and review of 2.1.1-2.1.4.
Actual column coverage, row counts, identifier gaps, RCID uniqueness, duplicates, and
parent issues remain unknown until that file is supplied. Missing optional columns will
limit later methods; missing RCID or company name, an empty export, repeated keys, or
hierarchy inconsistencies require investigation before any later matching work.
A company-reference snapshot does not establish identifier or parent validity during
2018-2025. No Part 2.2 matching, RCID acceptance, deduplication, or data export is implemented.

### Part 2.1C-A Exact Identifier Candidate Discovery

**Manual execution only.** This subsection is placed before Part 2.2 for organization,
but requires the existing completed Part 1, Part 2.1B, Part 2.2 and Part 2.3 variables
in the kernel. After a kernel restart, restore those prerequisites by manually running
the existing sections first, then return here. Saved output alone is not kernel state.
No previous section, candidate set, or RCID decision is changed.

Use exactly the 652 GVKEYs in `crosswalk_v1` and all historical observations in
`compustat_evidence`, without a latest-record or additional date filter. Direct GVKEY
uses exact trimmed strings, as in Part 1 (no zero-padding or zero removal). CIK uses
Part 2.3's `positive_digits`: digits only, positive, leading zeros removed in keys only.
Full CUSIP uses Part 2.3's `cusip_audit_key`: uppercase/trim, valid 8-character values
or 9-character values with a valid check digit. Equality includes the entire value;
8 and 9 characters never match via a prefix. This is stricter than the earlier baseline
shape-only CUSIP rule, whose original candidates remain unchanged.

All three methods search all targets independently. Shared identifiers retain every
target and every nonblank RCID; no contradictory identifier suppresses evidence.
Raw Revelio identifiers and parent/child RCIDs are retained for later review, not used
for parent substitution or hierarchy analysis. Historical Compustat raw values, dates,
and formats are retained once in `full_id_source_evidence`; candidate evidence links
to all applicable original `compustat_row` values. A direct GVKEY hit can exist even
without an annual observation, because membership in the established target universe
is sufficient for that method.

The scan reads 10,000 rows per chunk and only eight source columns. Unmatched rows
are discarded. Evidence is capped at 50,000 target/RCID/method/source-row records and
20 million retained raw-text characters (conservative initial limits, not an exact
RAM guarantee). Exceeding either limit aborts without publishing partial results;
review the limits before authorizing another run. No source deduplication or RCID
uniqueness is assumed. `revelio_source_row` is the one-based CSV data-record ordinal,
excluding the header, not a physical line number. Only the retained candidate subset
is grouped for reporting. Nothing is exported or hashed.

Reported firms, GVKEY?RCID pairs, method-specific evidence records, and distinct source
rows are different units. ?New pair? means absent from the union of Parts 1 and 2.2,
not necessarily a newly covered firm. Method firm counts overlap. These are snapshot
candidate associations, not verified corporate identities throughout 2018?2025.
Conflict resolution, RCID acceptance/rejection, and parent-child analysis remain pending.


In [ ]:
# 2.1C-A.1 ? Prerequisites, immutable comparison sets, and historical lookup evidence
from collections import defaultdict
import csv
import gzip

full_id_required = ["crosswalk_v1", "compustat_evidence", "saved_candidates",
                    "all_candidate_triples", "additional_local_candidates", "remaining_local",
                    "positive_digits", "cusip_audit_key", "revelio_full_audit"]
full_id_missing = [name for name in full_id_required if name not in globals()]
if full_id_missing:
    raise RuntimeError("Run the existing prerequisite sections first: " + ", ".join(full_id_missing))
full_id_targets = frozenset(crosswalk_v1.gvkey)
full_id_baseline_before = frozenset(saved_candidates)
full_id_previous_before = frozenset(all_candidate_triples)
full_id_additional_before = frozenset(additional_local_candidates[
    ["gvkey", "rcid", "method"]].itertuples(index=False, name=None))
full_id_unmatched_before = frozenset(remaining_local)
full_id_crosswalk_before = crosswalk_v1.copy(deep=True)
full_id_previous_pairs = frozenset((g, r) for g, r, m in full_id_previous_before)
assert len(full_id_targets) == len(crosswalk_v1) == 652 and crosswalk_v1.gvkey.is_unique
assert len({g for g, r, m in full_id_baseline_before}) == 565
assert full_id_previous_before == full_id_baseline_before | full_id_additional_before
assert len({g for g, r in full_id_previous_pairs}) == 608
assert full_id_unmatched_before == full_id_targets - {g for g, r in full_id_previous_pairs}
assert len(full_id_unmatched_before) == 44
assert revelio_full_audit["summary"]["rows"] == 33_961_589
assert crosswalk_v1.rcid.isna().all()

full_id_source_evidence = compustat_evidence[[
    "compustat_row", "gvkey_key", "gvkey", "conm", "cik", "cusip",
    "datadate", "fyear", "indfmt", "datafmt", "consol", "curcd",
]].copy()
assert full_id_source_evidence.compustat_row.is_unique
assert set(full_id_source_evidence.gvkey_key) <= full_id_targets
full_id_source_evidence["cik_comparison"] = full_id_source_evidence.cik.map(positive_digits)
full_id_source_evidence["cusip_comparison"] = full_id_source_evidence.cusip.map(
    lambda value: cusip_audit_key(value)[0])
full_id_lookup = {method: defaultdict(dict) for method in ["direct_gvkey", "cik", "full_cusip"]}
for full_id_g in sorted(full_id_targets):
    full_id_lookup["direct_gvkey"][full_id_g][full_id_g] = []
for full_id_row in full_id_source_evidence.itertuples(index=False):
    for full_id_method, full_id_key in [
        ("direct_gvkey", full_id_row.gvkey_key),
        ("cik", full_id_row.cik_comparison), ("full_cusip", full_id_row.cusip_comparison),
    ]:
        if full_id_key:
            full_id_lookup[full_id_method][full_id_key].setdefault(
                full_id_row.gvkey_key, []).append(int(full_id_row.compustat_row))
# Freeze row-reference lists once; candidate records share these compact tuples.
full_id_lookup = {m: {key: {g: tuple(rows) for g, rows in owners.items()}
                         for key, owners in mapping.items()}
                  for m, mapping in full_id_lookup.items()}
print("Targets:", len(full_id_targets), "| historical Compustat rows:", len(full_id_source_evidence))


In [ ]:
# 2.1C-A.2 ? Bounded full-reference scan; retains only target-matching evidence

def discover_full_identifier_candidates(path, lookup, chunksize=10_000,
                                        max_evidence=50_000, max_raw_characters=20_000_000):
    fields = ["rcid", "company", "gvkey", "cik", "cusip",
              "ultimate_parent_rcid", "child_rcid", "ticker"]
    with gzip.open(path, "rt", encoding="utf-8-sig", newline="") as handle:
        header = next(csv.reader(handle, strict=True), [])
    labels = [label.strip().lower() for label in header]
    if not labels or any(not x for x in labels) or len(set(labels)) != len(labels):
        raise ValueError("Empty or ambiguous CSV header; stop for review.")
    columns = dict(zip(labels, header))
    missing = set(fields) - set(columns)
    if missing:
        raise ValueError("Required evidence columns absent: " + ", ".join(sorted(missing)))
    evidence = []
    rows_scanned = raw_characters = blank_rcid_hits = 0
    output_columns = ["gvkey", "rcid", "method", "comparison_key", "revelio_source_row",
                      "compustat_rows"] + ["revelio_" + f + "_raw" for f in fields]
    with pd.read_csv(path, compression="gzip", encoding="utf-8-sig", dtype="string",
                     keep_default_na=False, na_filter=False, skip_blank_lines=False,
                     usecols=[columns[f] for f in fields], chunksize=chunksize,
                     on_bad_lines="error") as reader:
        for chunk in reader:
            chunk = chunk.rename(columns={columns[f]: f for f in fields}).reset_index(drop=True)
            keys = pd.DataFrame(index=chunk.index)
            keys["direct_gvkey"] = chunk.gvkey.str.strip()
            cik = chunk.cik.str.strip()
            keys["cik"] = cik.str.lstrip("0").where(cik.str.fullmatch(r"[0-9]+"), "")
            # Lookup keys already passed the existing full-CUSIP validator.
            # Exact equality to those keys also validates the matching reference value.
            keys["full_cusip"] = chunk.cusip.str.strip().str.upper()
            hit = pd.Series(False, index=chunk.index)
            for method in lookup:
                hit |= keys[method].isin(lookup[method])
            for position in chunk.index[hit]:
                raw = tuple(chunk.at[position, f] for f in fields)
                rcid = raw[0].strip()
                if not rcid:
                    blank_rcid_hits += 1
                    continue  # Count explicitly; a blank RCID cannot define a candidate pair.
                for method in lookup:
                    key = keys.at[position, method]
                    for gvkey, source_rows in lookup[method].get(key, {}).items():
                        next_characters = raw_characters + sum(len(value) for value in raw)
                        if len(evidence) >= max_evidence or next_characters > max_raw_characters:
                            raise RuntimeError("Candidate retention limit reached; no complete results published. "
                                               "Review accumulation before changing limits and rescanning.")
                        evidence.append((gvkey, rcid, method, key,
                                         rows_scanned + int(position) + 1, source_rows, *raw))
                        raw_characters = next_characters
            rows_scanned += len(chunk)
            del chunk, keys, cik, hit
    return pd.DataFrame.from_records(evidence, columns=output_columns), {
        "rows_scanned": rows_scanned, "identifier_hit_rows_without_rcid": blank_rcid_hits,
        "retained_evidence_records": len(evidence), "retained_raw_characters": raw_characters,
    }

full_id_evidence = full_id_scan_summary = full_id_pairs = None
full_id_status = "SCAN_IN_PROGRESS"
print("Scanning all gzip records in 10,000-row chunks; no intermediate results are final.", flush=True)
try:
    full_id_evidence, full_id_scan_summary = discover_full_identifier_candidates(
        revelio_full_company_path, full_id_lookup)
    if full_id_scan_summary["rows_scanned"] != revelio_full_audit["summary"]["rows"]:
        raise ValueError("Row count differs from Part 2.1B; investigate before reporting candidates.")
except BaseException:
    full_id_evidence = full_id_scan_summary = None
    full_id_status = "FAILED_OR_INTERRUPTED"
    raise
full_id_status = "SCAN_COMPLETE_REVIEW_REQUIRED"


In [ ]:
# 2.1C-A.3 Separate firm, pair, method, and source-row counts; preview new pairs
if full_id_status != "SCAN_COMPLETE_REVIEW_REQUIRED":
    raise RuntimeError("A complete scan is required before reporting.")
full_id_pairs = full_id_evidence.groupby(["gvkey", "rcid"], sort=True).agg(
    methods=("method", lambda values: tuple(sorted(set(values)))),
    source_rows=("revelio_source_row", "nunique"),
    evidence_records=("method", "size"),
).reset_index()
full_id_pairs["new_vs_previous_pairs"] = [
    (g, r) not in full_id_previous_pairs for g, r in full_id_pairs[["gvkey", "rcid"]].itertuples(index=False, name=None)]
full_id_firms = set(full_id_pairs.gvkey)
full_id_recovered = full_id_firms & full_id_unmatched_before
full_id_multiple = int(full_id_pairs.groupby("gvkey").rcid.nunique().gt(1).sum())
full_id_overview = pd.DataFrame([{
    "target_firms": len(full_id_targets), "firms_with_full_reference_candidates": len(full_id_firms),
    "previously_unmatched_44_with_candidates": len(full_id_recovered),
    "firms_with_multiple_full_reference_rcids": full_id_multiple,
    "full_reference_candidate_pairs": len(full_id_pairs),
    "new_pairs_vs_parts_1_and_2_2": int(full_id_pairs.new_vs_previous_pairs.sum()),
    "distinct_matching_source_rows": full_id_evidence.revelio_source_row.nunique(),
    "source_row_target_method_records": len(full_id_evidence),
}])
full_id_method_summary = full_id_evidence.groupby("method").agg(
    target_firms=("gvkey", "nunique"), source_rows=("revelio_source_row", "nunique"),
    evidence_records=("method", "size"),
).reindex(["direct_gvkey", "cik", "full_cusip"], fill_value=0)
full_id_method_summary["candidate_pairs"] = [
    len(full_id_evidence.loc[full_id_evidence.method.eq(m), ["gvkey", "rcid"]].drop_duplicates())
    for m in full_id_method_summary.index]
full_id_combinations = full_id_pairs.groupby("methods").size().rename("candidate_pairs").to_frame()
# Bounded preview of evidence for pairs absent from BOTH earlier candidate sets.
full_id_new_pair_keys = set(full_id_pairs.loc[full_id_pairs.new_vs_previous_pairs,
    ["gvkey", "rcid"]].itertuples(index=False, name=None))
full_id_new_mask = [(g, r) in full_id_new_pair_keys for g, r in
    full_id_evidence[["gvkey", "rcid"]].itertuples(index=False, name=None)]
full_id_preview = full_id_evidence.loc[full_id_new_mask].head(10)
assert frozenset(saved_candidates) == full_id_baseline_before
assert frozenset(all_candidate_triples) == full_id_previous_before
assert frozenset(additional_local_candidates[["gvkey", "rcid", "method"]].itertuples(
    index=False, name=None)) == full_id_additional_before
assert frozenset(remaining_local) == full_id_unmatched_before
pd.testing.assert_frame_equal(crosswalk_v1, full_id_crosswalk_before)
assert full_id_firms <= full_id_targets
assert not full_id_evidence.duplicated(["gvkey", "rcid", "method", "revelio_source_row"]).any()
print("Scan totals:", full_id_scan_summary)
with pd.option_context("display.max_columns", None, "display.max_colwidth", 100):
    display(full_id_overview)
    display(full_id_method_summary)
    display(full_id_combinations)
    display(full_id_preview)
print("Previously unmatched GVKEYs now having candidates:", sorted(full_id_recovered))
print("Preview source identifiers and dates are traceable via compustat_rows into full_id_source_evidence.")
print("Method counts overlap. Pair/source-row counts are not firm counts. No RCID accepted or rejected.")
print("STOP: candidate discovery only; historical identity, conflicts and hierarchy await review.")


### 2.2 Additional Local Matching

#### 2.2.1 Scope, verified baseline, and comparison rules

Run Part 1.2 first. The CIK mask repair fills missing Boolean results with `False`
and converts both operands to ordinary Boolean dtype before combining them; it does
not change which identifiers qualify. All **1,638** saved candidate-method triples
must equal the reconstruction, with **652 / 539 / 26 / 87** unchanged.

Part 2.1 is preserved as the earlier full-reference acquisition plan; its statements
that Part 2.2 was not implemented describe that preparation stage. This section now
uses only the existing ticker-filtered extract and does not depend on a full download.

Sequence: exact ticker, exact normalized name, then LinkedIn URL. Only firms still
without a candidate proceed to the next method, including when an earlier method yields
ambiguous or conflicting candidates. Candidate coverage is **not verified identity**.
No fuzzy matching, symbol punctuation rewriting, parent substitution, or RCID acceptance.

Ticker keys trim whitespace and standardize case but retain punctuation and suffixes
(e.g. `BRK.B`, `CA.1`). Name keys case-fold, normalize Unicode, replace punctuation with
spaces, collapse whitespace, and remove repeated terminal legal suffixes: Inc,
Incorporated, Corp, Corporation, Ltd, Limited, LLC, PLC, Co, Company. `L.L.C.` is treated
as LLC. Meaningful words such as Holdings, Bank, and Group are retained. Normalization
can create collisions; all RCIDs survive and conflicts remain visible.

In [ ]:
import re
import unicodedata
from urllib.parse import urlsplit

assert len(crosswalk_v1) == 652
assert len(saved_candidates) == len(reconstructed_candidates) == 1638
assert saved_candidates == reconstructed_candidates
original_candidate_firms = {g for g, r, m in saved_candidates}
original_unmatched = set(no_local_candidate_review["gvkey"])
assert len(original_candidate_firms) == 565 and len(original_unmatched) == 87
assert not original_candidate_firms & original_unmatched
baseline_candidate_snapshot = frozenset(saved_candidates)
baseline_crosswalk_snapshot = crosswalk_v1.copy(deep=True)

def text_key(value):
    return "" if pd.isna(value) else str(value).strip()

def ticker_key(value):
    return text_key(value).upper()

def name_key(value):
    text = unicodedata.normalize("NFKC", text_key(value)).casefold()
    text = re.sub(r"\bl\s*\.\s*l\s*\.\s*c\s*\.?", "llc", text)
    text = "".join(" " if unicodedata.category(ch).startswith("P") else ch for ch in text)
    words = text.split()
    suffixes = {"inc", "incorporated", "corp", "corporation", "ltd", "limited", "llc", "plc", "co", "company"}
    while words and words[-1] in suffixes:
        words.pop()
    return " ".join(words)

def linkedin_key(value):
    text = text_key(value)
    if not text:
        return ""
    parsed = urlsplit(text if "://" in text else "https://" + text)
    host = (parsed.hostname or "").lower()
    if host != "linkedin.com" and not host.endswith(".linkedin.com"):
        return ""
    parts = parsed.path.strip("/").split("/")
    if len(parts) != 2 or parts[0].lower() != "company" or not re.fullmatch(r"[A-Za-z0-9_-]+", parts[1]):
        return ""
    return "linkedin.com/company/" + parts[1].lower()

# Focused safeguards: missing keys never match and suffix/punctuation rules are explicit.
assert name_key(pd.NA) == ticker_key(pd.NA) == linkedin_key(pd.NA) == ""
assert name_key(" Acme, Corporation ") == name_key("ACME Inc.") == "acme"
assert name_key("Acme L.L.C.") == "acme"
assert name_key("Acme Holdings Inc.") != name_key("Acme Inc.")
assert ticker_key("BRK.B") != ticker_key("BRK-B")
assert linkedin_key("https://www.linkedin.com/company/acme/?trk=x") == "linkedin.com/company/acme"
assert linkedin_key("https://linkedin.com.evil.test/company/acme") == ""
assert linkedin_key("https://linkedin.com/in/person") == ""

#### 2.2.2 Dated source evidence and ticker-reuse diagnostics

Compustat evidence retains every observed annual record, original identifiers, reporting
date, fiscal year, and format. Reporting dates do not prove ticker-validity intervals.
CRSP daily tickers are attached to GVKEY only through Notebook 01's preserved P/C link
history and only on trading dates inside each dated membership/link overlap. LS links
remain review flags. All daily dates and original CSV data-row numbers are retained in
`crsp_ticker_observations`; each compressed evidence record references those rows and
retains the complete observed-date tuple. First/last observations do not imply uninterrupted
validity. No J-only or undated PERMNO-to-GVKEY join is used.

Reuse flags cover tickers shared by multiple Compustat GVKEYs in the full local annual
extract or multiple CRSP PERMCOs in the local daily extract. These are possible reuse or
conflict signals, not a determination of sequential versus simultaneous use. The local
Revelio snapshot has no ticker-validity dates, so temporal identity remains unverified.

In [ ]:
# Every record in this long table is source evidence, not a selected company identifier.
source_records = []
for row in compustat_evidence.to_dict("records"):
    source_records.append({
        "gvkey": row["gvkey_key"], "source": "compustat", "source_row": row["compustat_row"],
        "company_name": row["conm"], "ticker": row["tic"], "cik": row["cik"], "cusip": row["cusip"],
        "datadate": row["datadate"], "fyear": row["fyear"], "indfmt": row["indfmt"],
        "datafmt": row["datafmt"], "consol": row["consol"], "curcd": row["curcd"],
        "linktype": "", "linkedin_url": "",
    })
crsp_path_local = project_root / "data/raw/crsp/crsp_sp500_20261022.csv"
crsp_ticker_observations = pd.read_csv(
    crsp_path_local, usecols=["PERMNO", "PERMCO", "Ticker", "DlyCalDt", "CUSIP", "HdrCUSIP"],
    dtype="string", keep_default_na=False, na_values=[""],
)
crsp_ticker_observations["source_row"] = range(1, len(crsp_ticker_observations) + 1)
crsp_ticker_observations["permno_key"] = crsp_ticker_observations["PERMNO"].map(text_key)
crsp_ticker_observations["ticker_key"] = crsp_ticker_observations["Ticker"].map(ticker_key)
crsp_ticker_observations["date"] = pd.to_datetime(crsp_ticker_observations["DlyCalDt"], errors="coerce")
crsp_groups = {key: group for key, group in crsp_ticker_observations.groupby("permno_key")}
for firm in crosswalk_v1.itertuples(index=False):
    for link in json.loads(firm.historical_links_json):
        assert link["linkprim"] in ("P", "C")
        observations = crsp_groups.get(text_key(link["permno"]))
        if observations is None:
            continue
        eligible = observations.loc[observations["date"].between(
            pd.Timestamp(link["overlap_start"]), pd.Timestamp(link["overlap_end"]))]
        for keys, group in eligible.groupby(["Ticker", "CUSIP", "HdrCUSIP", "PERMCO"], dropna=False):
            ticker, cusip, hdrcusip, permco = keys
            source_records.append({
                "gvkey": firm.gvkey, "source": "crsp", "source_row": tuple(group["source_row"]),
                "company_name": "", "ticker": ticker, "cik": "", "cusip": cusip,
                "hdr_cusip": hdrcusip, "permno": link["permno"], "permco": permco,
                "observed_dates": tuple(sorted(group["DlyCalDt"].unique())),
                "first_observed": group["DlyCalDt"].min(), "last_observed": group["DlyCalDt"].max(),
                "membership_row": link["membership_row"], "ccm_row": link["ccm_row"],
                "membership_start_source": link["membership_start_source"],
                "membership_end_source": link["membership_end_source"],
                "link_start_source": link["link_start_source"], "link_end_source": link["link_end_source"],
                "overlap_start": link["overlap_start"], "overlap_end": link["overlap_end"],
                "linktype": link["linktype"], "linkedin_url": "",
            })
local_source_evidence = pd.DataFrame(source_records)
for column, normalizer in [("ticker", ticker_key), ("company_name", name_key), ("linkedin_url", linkedin_key)]:
    local_source_evidence[column + "_key"] = local_source_evidence[column].map(normalizer)

# Reuse scope extends beyond the 87 targets; source frames remain unchanged.
all_comp_tickers = pd.DataFrame({"ticker_key": compustat_source["tic"].map(ticker_key),
                                 "gvkey": compustat_source["gvkey"].map(text_key)})
comp_ticker_multiplicity = all_comp_tickers.loc[
    all_comp_tickers.ticker_key.ne("") & all_comp_tickers.gvkey.ne("")
].groupby("ticker_key").gvkey.nunique()
crsp_ticker_multiplicity = crsp_ticker_observations.loc[
    crsp_ticker_observations.ticker_key.ne("")
].groupby("ticker_key").PERMCO.nunique()
reuse_tickers = set(comp_ticker_multiplicity[comp_ticker_multiplicity.gt(1)].index) | set(
    crsp_ticker_multiplicity[crsp_ticker_multiplicity.gt(1)].index)
local_reference = revelio_local_source.copy()
local_reference.columns = local_reference.columns.str.lower()
local_reference["revelio_source_row"] = range(1, len(local_reference) + 1)
local_reference["rcid_key"] = local_reference.rcid.map(text_key)
for column, normalizer in [("ticker", ticker_key), ("company", name_key), ("linkedin_url", linkedin_key)]:
    local_reference[column + "_key"] = local_reference[column].map(normalizer)
local_reference = local_reference.loc[local_reference.rcid_key.ne("")].copy()
print("Dated evidence records:", len(local_source_evidence), "| CRSP daily observations:", len(crsp_ticker_observations))
print("Possible reused/conflicting tickers in available source scope:", len(reuse_tickers))

#### 2.2.3 Candidate generation and review flags

`local_candidate_evidence` retains every source-evidence / Revelio-row pairing, including
raw identifier strings, dates, source-row references, hierarchy, method, and comparison key.
`additional_local_candidates` is one row per GVKEY/RCID/method and points to all evidence
rows. Multiple evidence rows are not multiple firm matches. Unique means one distinct
candidate RCID, not an accepted identity. A contradictory nonblank Revelio GVKEY or
nonoverlapping observed CIK/full-CUSIP sets is flagged; missing comparison evidence is
reported separately. Name disagreement, parent scope, shared RCIDs, ticker reuse, and
LS links remain review cues. None of these flags silently discards a candidate.

In [ ]:
firm_identifier_sets = {}
for gvkey, group in compustat_evidence.groupby("gvkey_key"):
    firm_identifier_sets[gvkey] = {
        "cik": set(group["cik_for_comparison"].dropna()),
        "cusip": set(group["cusip_for_comparison"].dropna()),
        "names": {name_key(v) for v in group["conm"] if name_key(v)},
    }

def cik_key_local(value):
    value = text_key(value)
    return (value.lstrip("0") or "") if re.fullmatch(r"[0-9]+", value) else ""

def cusip_key_local(value):
    value = text_key(value).upper()
    return value if re.fullmatch(r"[0-9A-Z*@#]{8,9}", value) else ""

baseline_rcid_owners = {}
for gvkey, rcid, method in saved_candidates:
    baseline_rcid_owners.setdefault(rcid, set()).add(gvkey)
local_evidence_records = []
local_method_reports = []
remaining_local = set(original_unmatched)

def run_local_method(method, left_key, right_key, available=True):
    global remaining_local
    before = set(remaining_local)
    left = local_source_evidence.loc[local_source_evidence.gvkey.isin(before)]
    reference_index = {key: group for key, group in local_reference.loc[
        local_reference[right_key].ne("")].groupby(right_key)}
    new_pairs = set()
    if available:
        for evidence in left.to_dict("records"):
            key = evidence[left_key]
            if not key or key not in reference_index:
                continue
            for ref in reference_index[key].to_dict("records"):
                gvkey, rcid = evidence["gvkey"], ref["rcid_key"]
                known = firm_identifier_sets.get(gvkey, {"cik": set(), "cusip": set(), "names": set()})
                rcik, rcusip = cik_key_local(ref["cik"]), cusip_key_local(ref["cusip"])
                # Include eligible dated CRSP full CUSIPs for firms missing annual evidence.
                crsp_cusips = {cusip_key_local(v) for v in local_source_evidence.loc[
                    local_source_evidence.gvkey.eq(gvkey), "cusip"] if cusip_key_local(v)}
                source_cusips = known["cusip"] | crsp_cusips
                record = {
                    "evidence_id": len(local_evidence_records) + 1, "gvkey": gvkey, "rcid": rcid,
                    "method": method, "comparison_key": key, "source_evidence": evidence.copy(),
                    "revelio_evidence": ref.copy(),
                    "gvkey_conflict": bool(text_key(ref["gvkey"]) and text_key(ref["gvkey"]) != gvkey),
                    "cik_conflict": bool(rcik and known["cik"] and rcik not in known["cik"]),
                    "cusip_conflict": bool(rcusip and source_cusips and rcusip not in source_cusips),
                    "cik_comparison_unavailable": not bool(rcik and known["cik"]),
                    "cusip_comparison_unavailable": not bool(rcusip and source_cusips),
                    "name_disagreement": bool(known["names"] and name_key(ref["company"]) not in known["names"]),
                    "ticker_reuse_or_conflict": method == "ticker" and key in reuse_tickers,
                    "parent_scope_review": bool(text_key(ref["ultimate_parent_rcid"]) and text_key(ref["ultimate_parent_rcid"]) != rcid),
                    "ls_link_review": text_key(evidence.get("linktype")) == "LS",
                    "rcid_shared_with_baseline_firm": bool(baseline_rcid_owners.get(rcid, set()) - {gvkey}),
                    "historical_validity_unverified": True,
                }
                local_evidence_records.append(record)
                new_pairs.add((gvkey, rcid))
    counts = pd.Series([g for g, r in new_pairs], dtype="string").value_counts()
    received = set(counts.index)
    remaining_local -= received
    report = {"method": method, "availability": "available" if available else "unavailable: no reliable firm-side LinkedIn identifiers",
              "unmatched_before": len(before), "firms_with_new_candidates": len(received),
              "unique_candidate_firms": int(counts.eq(1).sum()), "ambiguous_candidate_firms": int(counts.gt(1).sum()),
              "still_unmatched": len(remaining_local), "cumulative_candidate_firms": 652 - len(remaining_local),
              "candidate_coverage_pct": round(100 * (652 - len(remaining_local)) / 652, 2)}
    local_method_reports.append(report)
    assert len(before) == len(received) + len(remaining_local)
    display(pd.DataFrame([report]))

#### 2.2.4 Exact ticker matching

Search all observed Compustat and dated P/C-linked CRSP ticker keys. Retain every
candidate RCID, including ticker collisions and conflicting identities.

In [ ]:
run_local_method("ticker", "ticker_key", "ticker_key")

#### 2.2.5 Exact normalized company-name matching

Search only firms left without ticker candidates. Compare Compustat company names to
the Revelio record's own company name; do not match to a parent's or child's name.

In [ ]:
run_local_method("normalized_company_name", "company_name_key", "company_key")

#### 2.2.6 LinkedIn URL availability and matching

The Compustat and CRSP files have no company LinkedIn URL field, and the saved crosswalk
has no independent firm-side LinkedIn evidence. A URL copied from a candidate Revelio
row would be circular evidence. Therefore this method is unavailable for this run.
The URL normalizer permits only company paths on LinkedIn, strips query/fragment and
trailing slash, and standardizes host/case; it never invents a URL from a name.

In [ ]:
linkedin_available = bool(local_source_evidence["linkedin_url_key"].ne("").any())
assert not linkedin_available, "New firm-side LinkedIn evidence requires provenance review."
run_local_method("linkedin_url", "linkedin_url_key", "linkedin_url_key", available=linkedin_available)
local_matching_summary = pd.DataFrame(local_method_reports)
display(local_matching_summary)

#### 2.2.7 Complete candidate and unmatched review tables

The candidate table summarizes all evidence without choosing an RCID. The unmatched
table includes every remaining GVKEY, original recovered names/identifiers, dated-source
references, missing-evidence flags, and method-specific reasons. Empty tuples mean no
observed value, not absence of the company. Unknown names are labeled rather than guessed.

In [ ]:
local_candidate_evidence = pd.DataFrame(local_evidence_records)
flag_columns = [col for col in local_candidate_evidence.columns if col not in
                ["evidence_id", "gvkey", "rcid", "method", "comparison_key", "source_evidence", "revelio_evidence"]]
candidate_rows = []
for (gvkey, rcid, method), group in local_candidate_evidence.groupby(["gvkey", "rcid", "method"]):
    candidate_rows.append({"gvkey": gvkey, "rcid": rcid, "method": method,
                           "evidence_ids": tuple(group.evidence_id),
                           "comparison_keys": tuple(sorted(group.comparison_key.unique())),
                           **{flag: bool(group[flag].any()) for flag in flag_columns}})
additional_local_candidates = pd.DataFrame(candidate_rows)
new_counts = additional_local_candidates.groupby("gvkey").rcid.nunique()
additional_local_candidates["ambiguous_rcid"] = additional_local_candidates.gvkey.map(new_counts).gt(1)
new_rcid_owners = additional_local_candidates.groupby("rcid").gvkey.nunique()
additional_local_candidates["rcid_shared_with_new_firms"] = additional_local_candidates.rcid.map(new_rcid_owners).gt(1)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(additional_local_candidates.drop(columns="evidence_ids"))

def original_values(values):
    return tuple(sorted({str(v) for v in values if text_key(v)}))

review_rows = []
for gvkey in sorted(remaining_local):
    source = local_source_evidence.loc[local_source_evidence.gvkey.eq(gvkey)]
    annual = compustat_evidence.loc[compustat_evidence.gvkey_key.eq(gvkey)]
    names = original_values(source.company_name)
    tickers = original_values(source.ticker)
    ciks = original_values(source.cik)
    cusips = original_values(source.cusip)
    usable_ticker = source.ticker_key.ne("").any()
    usable_name = source.company_name_key.ne("").any()
    review_rows.append({"gvkey": gvkey, "company_names": names,
        "name_status": "observed" if names else "unavailable in local Compustat/CRSP schema",
        "cik": ciks, "cusip": cusips, "tickers": tickers,
        "compustat_rows": tuple(annual.compustat_row),
        "compustat_reporting_dates": original_values(annual.datadate),
        "crsp_permnos": original_values(source.loc[source.source.eq("crsp"), "permno"]),
        "crsp_dates": tuple(sorted({d for v in source.loc[source.source.eq("crsp"), "observed_dates"] for d in v})),
        "ticker_reason": "no exact ticker in local Revelio with nonblank RCID" if usable_ticker else "no usable source ticker",
        "name_reason": "no exact normalized name in local Revelio with nonblank RCID" if usable_name else "no usable source company name",
        "linkedin_reason": "unavailable: no independently sourced company LinkedIn URL",
        "missing_company_name": not bool(names), "missing_cik": not bool(ciks),
        "missing_full_cusip": not bool(cusips), "missing_ticker": not bool(tickers),
        "next_evidence_needed": "full Revelio reference / historical aliases; independently sourced LinkedIn company URL",
    })
remaining_unmatched_review = pd.DataFrame(review_rows)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
    display(remaining_unmatched_review.drop(columns=["crsp_dates", "compustat_rows"]))
print("Complete dates and source-row references remain in remaining_unmatched_review and local_source_evidence.")

#### 2.2.8 Preservation, uniqueness, and coverage validation

Build a separate one-row-per-firm candidate review index. Original candidate JSON is
retained and every candidate-method triple must survive. Accepted RCID and decision
fields remain null. This is an in-memory review index, not Crosswalk v2 or an export.

In [ ]:
new_candidate_triples = set(additional_local_candidates[["gvkey", "rcid", "method"]].itertuples(index=False, name=None))
all_candidate_triples = set(saved_candidates) | new_candidate_triples
assert set(saved_candidates) == set(baseline_candidate_snapshot)
assert {t for t in all_candidate_triples if t[0] in original_candidate_firms} == set(saved_candidates)
assert {g for g, r, m in new_candidate_triples}.issubset(original_unmatched)
assert not additional_local_candidates.duplicated(["gvkey", "rcid", "method"]).any()
assert additional_local_candidates.groupby("gvkey").method.nunique().le(1).all()
assert remaining_local == original_unmatched - set(additional_local_candidates.gvkey)
assert len(remaining_unmatched_review) == len(remaining_local)
assert remaining_unmatched_review.gvkey.is_unique

local_firm_review_index = baseline_review.copy(deep=True)
local_firm_review_index["candidate_rcids"] = [tuple(sorted({r for g, r, m in all_candidate_triples if g == gvkey}))
                                            for gvkey in local_firm_review_index.gvkey]
local_firm_review_index["new_method"] = local_firm_review_index.gvkey.map(
    additional_local_candidates.drop_duplicates("gvkey").set_index("gvkey").method)
local_firm_review_index["original_candidate_json"] = local_firm_review_index.gvkey.map(
    crosswalk_v1.set_index("gvkey").revelio_candidates_json)
local_firm_review_index["accepted_rcid"] = pd.Series(pd.NA, index=local_firm_review_index.index, dtype="string")
assert len(local_firm_review_index) == 652 and local_firm_review_index.gvkey.is_unique
assert local_firm_review_index.accepted_rcid.isna().all()
assert int(local_firm_review_index.candidate_rcids.map(len).gt(0).sum()) == 652 - len(remaining_local)
pd.testing.assert_frame_equal(crosswalk_v1, baseline_crosswalk_snapshot)
pd.testing.assert_frame_equal(crosswalk_v1, pd.read_parquet(input_paths["crosswalk_v1"]))
assert crosswalk_v1.rcid.isna().all() and crosswalk_v1.decision_basis.isna().all()
# Verify original raw annual identifiers survived in recovered evidence.
for column in ["conm", "cik", "cusip", "tic"]:
    original = compustat_source.iloc[compustat_evidence.compustat_row.to_numpy() - 1][column].reset_index(drop=True)
    pd.testing.assert_series_equal(compustat_evidence[column].reset_index(drop=True), original)
print("PASS: 652 unique firms; 565 original candidate firms and 1,638 triples preserved; zero accepted RCIDs.")
print("New candidate firms:", len(original_unmatched) - len(remaining_local))
print("Candidate coverage:", 652 - len(remaining_local), "/ 652; remaining unmatched:", len(remaining_local))

#### 2.2.9 Findings

The executable findings below are generated from the validated local run. The count of
new candidates includes ambiguous and contradictory matches and must not be described
as accepted coverage. No source file, Notebook 01, v1 crosswalk, or membership bridge
is written. No full-reference download, web search, or final Crosswalk v2 export occurs.

In [ ]:
from IPython.display import Markdown
findings_lines = ["**Validated local matching findings**", ""]
for report in local_method_reports:
    findings_lines.append(f"- {report['method']}: {report['firms_with_new_candidates']} additional firms "
                          f"({report['unique_candidate_firms']} single-RCID, {report['ambiguous_candidate_firms']} multiple-RCID); "
                          f"{report['still_unmatched']} remain. {report['availability']}.")
conflict_mask = additional_local_candidates[["gvkey_conflict", "cik_conflict", "cusip_conflict"]].any(axis=1)
findings_lines += [
    f"- Candidate coverage is **{652 - len(remaining_local)}/652 ({100*(652-len(remaining_local))/652:.2f}%)**; "
    f"**{len(remaining_local)} of the original 87 remain unmatched**. Accepted RCIDs remain zero.",
    f"- {additional_local_candidates.loc[conflict_mask, 'gvkey'].nunique()} newly covered firms have at least one "
    "candidate with a GVKEY, CIK, or full-CUSIP conflict. These require identity review; ticker equality alone is weak evidence.",
]
for row in remaining_unmatched_review.itertuples(index=False):
    missing = [label for flag, label in [(row.missing_company_name, "company name"), (row.missing_cik, "CIK"),
                                        (row.missing_full_cusip, "full CUSIP"), (row.missing_ticker, "ticker")] if flag]
    if missing:
        label = " / ".join(row.company_names) if row.company_names else "name unavailable"
        findings_lines.append(f"- Insufficient source fields: GVKEY {row.gvkey} ({label}) lacks {', '.join(missing)}.")
findings_lines += [
    "- All remaining firms lack independent firm-side LinkedIn identifiers. Missing source names cannot be inferred from their GVKEYs.",
    "- The local searches are exhausted under these exact rules. A fuller Revelio reference is the next coverage source for remaining firms, "
    "but recovery is not guaranteed. Needed fields: RCID, company name and historical aliases, CIK, full CUSIP/ISIN, ticker with exchange "
    "and effective dates, company LinkedIn URL, GVKEY where available, and parent RCID/name with effective dates.",
    "- For firms missing Compustat evidence, also obtain a dated GVKEY/PERMNO-to-company-name/CIK reference and an independently verified "
    "LinkedIn company URL. The existing CRSP dates support ticker observations only; the undated Revelio snapshot cannot resolve ticker reuse "
    "or historical parent ownership. No additional data were requested or downloaded in this section.",
]
display(Markdown("\n".join(findings_lines)))

### 2.3 Candidate Quality Audit

#### 2.3.1 Scope and audit contract

Audit all 73 new GVKEY?RCID pairs from Part 2.2, using only local evidence. Earlier
candidate sets, conflict flags, and unmatched firms remain unchanged. Corrected diagnostic
columns below **supersede the interpretation**, not the stored Part 2.2 flags. No candidate
is removed, accepted, or assigned to the crosswalk.

Notebook 01 Parts 5.1?5.5 compare full strings and explicitly leave CUSIP length differences
unresolved. Part 2.2 also compares strings of length 8 or 9 without allowing a valid
8/9 equivalence. Its `cusip_conflict` is therefore a full-string disagreement flag, not
necessarily a contradictory company identifier. Its source CUSIP pool includes annual
Compustat and dated CRSP observations, despite the variable name `crsp_cusips`.

This audit distinguishes format equivalence, actual observed security-identifier variation,
missing/invalid identifiers, and disagreement with **all** valid observed CUSIPs. Even real
CUSIP disagreement may reflect different securities or periods: it is not by itself proof
of different companies. No merger, acquisition, rename, or parent history is invented.

In [ ]:
audit_candidates_before = additional_local_candidates.copy(deep=True)
audit_unmatched_before = remaining_unmatched_review.copy(deep=True)
audit_firms_before = local_firm_review_index.copy(deep=True)
audit_triples_before = frozenset(all_candidate_triples)
assert len(crosswalk_v1) == 652 and len(original_candidate_firms) == 565
assert len(saved_candidates) == 1638
assert len(additional_local_candidates) == 73
assert additional_local_candidates.gvkey.nunique() == 43
assert additional_local_candidates.method.eq("ticker").all()
assert additional_local_candidates.cusip_conflict.all()
assert int(additional_local_candidates.groupby("gvkey").rcid.nunique().gt(1).sum()) == 16
assert len(remaining_local) == len(remaining_unmatched_review) == 44
assert int(local_firm_review_index.candidate_rcids.map(len).gt(0).sum()) == 608
print("Reconciled: 652 firms; 565 original candidate firms; 73 new pairs / 43 firms; 608 covered; 44 unmatched.")

#### 2.3.2 CUSIP validity, raw values, and length distributions

Source strings are retained unchanged. Separate keys trim whitespace and uppercase.
An 8-character value is eligible only if it has eight CUSIP characters; a 9-character
value must additionally have a numeric, correct check digit. This is a structural check,
not verification against an identifier registry. Eight-character values lack a check
digit and cannot be checksum-verified. Never truncate an invalid 9-character value.

`cusip8_format_agreement` means valid values agree on the first eight characters with
an 8-versus-9 length difference. `cusip8_agreement_9_vs_9` is separately supported, but
with valid check digits equal first-eight characters normally imply identical full values.
Missing or invalid evidence never becomes a conflict. Mixed agreement/disagreement is
retained for review instead of selecting one preferred identifier.

The length table uses explicit denominators: all downloaded source rows, then rows for
the 43 audited firms / 73 RCIDs. CRSP historical values are shown separately. The raw-value
ledger retains each distinct source value and its validity status; row references and all
dates remain in the source-evidence tables.

In [ ]:
def cusip_audit_key(value):
    value = text_key(value).upper()
    if not value:
        return "", "missing"
    if not re.fullmatch(r"[0-9A-Z*@#]{8}([0-9])?", value):
        return "", "invalid_shape"
    if len(value) == 8:
        return value, "valid8_no_check_digit"
    char_values = {"*": 36, "@": 37, "#": 38}
    total = 0
    for pos, char in enumerate(value[:8]):
        number = int(char) if char.isdigit() else ord(char) - ord("A") + 10 if char.isalpha() else char_values[char]
        product = number * (1 if pos % 2 == 0 else 2)
        total += product // 10 + product % 10
    if int(value[-1]) != (10 - total % 10) % 10:
        return "", "invalid_check_digit"
    return value, "valid9_check_digit"

def compare_cusips(left_values, right_values):
    left = {cusip_audit_key(v)[0] for v in left_values} - {""}
    right = {cusip_audit_key(v)[0] for v in right_values} - {""}
    if not left or not right:
        return "missing_or_invalid", (), ()
    exact = left & right
    pairs = {(a, b) for a in left for b in right if a[:8] == b[:8]}
    format_pairs = {(a, b) for a, b in pairs if len(a) != len(b)}
    unmatched_right = tuple(sorted(b for b in right if not any(a[:8] == b[:8] for a in left)))
    if exact:
        status = "exact_full_cusip_agreement"
    elif format_pairs:
        status = "cusip8_format_agreement"
    elif pairs:
        status = "cusip8_agreement_9_vs_9"
    else:
        status = "disagreement_all_valid_observed"
    if pairs and unmatched_right:
        status = "mixed_agreement_and_disagreement"
    return status, tuple(sorted(pairs)), unmatched_right

# Regression cases: format difference must not be mislabeled as genuine disagreement.
assert compare_cusips(["03783310"], ["037833100"])[0] == "cusip8_format_agreement"
assert compare_cusips(["037833100"], ["037833100"])[0] == "exact_full_cusip_agreement"
assert compare_cusips(["037833100"], ["594918104"])[0] == "disagreement_all_valid_observed"
assert cusip_audit_key("037833109")[1] == "invalid_check_digit"
assert compare_cusips(["037833109"], ["037833100"])[0] == "missing_or_invalid"
assert compare_cusips([pd.NA], ["037833100"])[0] == "missing_or_invalid"
assert compare_cusips(["037833100"], ["037833100", "594918104"])[0] == "mixed_agreement_and_disagreement"

audit_gvkeys = set(additional_local_candidates.gvkey)
audit_rcids = set(additional_local_candidates.rcid)
audit_source = local_source_evidence.loc[local_source_evidence.gvkey.isin(audit_gvkeys)].copy()
audit_reference = local_reference.loc[local_reference.rcid_key.isin(audit_rcids)].copy()
length_rows = []
for label, values in [
    ("Compustat full extract: annual rows", compustat_source.cusip),
    ("Revelio filtered extract: source rows", revelio_local_source.cusip),
    ("Compustat audited 43 firms: annual rows", compustat_evidence.loc[compustat_evidence.gvkey_key.isin(audit_gvkeys), "cusip"]),
    ("Revelio audited 73 RCIDs: source rows", audit_reference.cusip),
    ("CRSP audited firms: dated evidence groups", audit_source.loc[audit_source.source.eq("crsp"), "cusip"]),
]:
    for (length, validity), count in pd.DataFrame({
        "length": values.map(lambda v: len(text_key(v))),
        "validity": values.map(lambda v: cusip_audit_key(v)[1]),
    }).value_counts().items():
        length_rows.append({"scope": label, "source_rows_or_groups": len(values),
                            "length": length, "validity": validity, "count": count})
cusip_length_distribution = pd.DataFrame(length_rows)
display(cusip_length_distribution)
raw_cusip_rows = []
for (gvkey, source), group in audit_source.groupby(["gvkey", "source"]):
    for value in original_values(group.cusip):
        raw_cusip_rows.append({"side": source, "entity_key": gvkey, "raw_cusip": value,
                               "length": len(text_key(value)), "validity": cusip_audit_key(value)[1]})
for rcid, group in audit_reference.groupby("rcid_key"):
    for value in original_values(group.cusip):
        raw_cusip_rows.append({"side": "revelio", "entity_key": rcid, "raw_cusip": value,
                               "length": len(text_key(value)), "validity": cusip_audit_key(value)[1]})
raw_cusip_audit = pd.DataFrame(raw_cusip_rows)
# Raw values for every candidate are displayed in the complete candidate table below.
print("Distinct source/entity/raw-CUSIP ledger records:", len(raw_cusip_audit))

#### 2.3.3 Dated histories and locally supported relationships

The complete history table below groups identical observed identities, preserving original
annual or trading-date ranges, observation counts, and source-row references. The full
date tuples remain in `audit_history_evidence` and the Part 2.2 evidence. A date range is
not an assertion of continuous identifier validity. CRSP `HdrCUSIP` is retained for
context, but is not substituted for the dated `CUSIP`.

A change of CUSIP8 on successive dated observations of the **same PERMNO** is reported as
an observed security-identifier transition. Several CUSIP8s across different securities
or between Compustat and CRSP are only variation, not proof of a time sequence, rename,
acquisition, or merger. Compustat company-name variation is similarly diagnostic.
Revelio parents and largest-child fields are undated snapshot assertions only.

In [ ]:
audit_history_evidence = audit_source.copy(deep=True)
history_rows = []
for gvkey, group in audit_source.groupby("gvkey"):
    annual = group.loc[group.source.eq("compustat")]
    for keys, rows in annual.groupby(["company_name", "ticker", "cik", "cusip"], dropna=False):
        history_rows.append({"gvkey": gvkey, "source": "compustat", "name_ticker_cik_cusip": keys,
                            "date_start": rows.datadate.min(), "date_end": rows.datadate.max(),
                            "observations": len(rows), "permno": "", "source_rows": tuple(rows.source_row)})
    for row in group.loc[group.source.eq("crsp")].to_dict("records"):
        history_rows.append({"gvkey": gvkey, "source": "crsp", "name_ticker_cik_cusip":
                            ("", row["ticker"], "", row["cusip"]), "date_start": row["first_observed"],
                            "date_end": row["last_observed"], "observations": len(row["observed_dates"]),
                            "permno": row["permno"], "source_rows": row["source_row"]})
audit_identity_histories = pd.DataFrame(history_rows)
transition_rows = []
for gvkey, group in audit_source.loc[audit_source.source.eq("crsp")].groupby("gvkey"):
    source_rows = sorted({int(i) for indices in group.source_row for i in indices})
    observations = crsp_ticker_observations.iloc[[i-1 for i in source_rows]].copy()
    observations["cusip8_audit"] = observations.CUSIP.map(lambda v: cusip_audit_key(v)[0][:8])
    for permno, security in observations.sort_values("date").groupby("PERMNO"):
        valid = security.loc[security.cusip8_audit.ne("")]
        previous = None
        for row in valid.itertuples():
            if previous is not None and previous.cusip8_audit != row.cusip8_audit:
                transition_rows.append({"gvkey": gvkey, "permno": permno,
                    "previous_date": previous.DlyCalDt, "next_date": row.DlyCalDt,
                    "previous_cusip": previous.CUSIP, "next_cusip": row.CUSIP,
                    "previous_source_row": previous.source_row, "next_source_row": row.source_row})
            previous = row
cusip_historical_transitions = pd.DataFrame(transition_rows, columns=["gvkey", "permno", "previous_date", "next_date",
    "previous_cusip", "next_cusip", "previous_source_row", "next_source_row"])
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
    display(audit_identity_histories.drop(columns="source_rows"))
    display(cusip_historical_transitions)

#### 2.3.4 Candidate-level comparisons and conservative classification

All source and Revelio records for each pair participate. CIK keys remove leading zeros
only from positive digit strings. GVKEY comparison uses positive digits with leading-zero
normalization in keys only. Names retain raw strings and the exact normalized keys from
Part 2.2; shared non-generic words (excluding sector words such as petroleum, oil, mining, and energy) are merely historical-review cues, **not a new match**.

Decision order (diagnostic, never acceptance):

1. `needs_historical_review` for conflicting identifiers with a shared distinctive name
   token, or mixed identifier agreement/disagreement. A shared token does not establish a relationship.
2. `likely_false_positive` when valid CUSIPs disagree with every observed security and
   another comparable identifier or the exact normalized name also disagrees.
3. `ambiguous_multiple_rcids` for remaining multiple-RCID cases; multiplicity also remains
   an independent flag on every row, even when another category is more informative.
4. `strong_candidate` requires CIK/GVKEY corroboration plus name or CUSIP corroboration,
   with no listed identifier contradiction. CUSIP8 alone can never produce this category.
5. `needs_historical_review` for other positive but incomplete evidence;
   `insufficient_evidence` otherwise.

Credible requiring final review is defined separately: corroborating CIK or GVKEY plus
name/CUSIP support and no contradiction. A name-token cue alone is **not credible evidence**.
P1 review goes to historical/mixed cases, source gaps, and multiple-RCID groups; P2 to
otherwise strong candidates and P3 to remaining well-supported likely false positives.
No category resolves an actual multiple-RCID decision; all 16 original groups remain unadjudicated.

In [ ]:
def positive_digits(value):
    value = text_key(value)
    return (value.lstrip("0") or "") if re.fullmatch(r"[0-9]+", value) else ""

def identifier_comparison(left, right, normalizer):
    a = {normalizer(v) for v in left} - {""}
    b = {normalizer(v) for v in right} - {""}
    if not a or not b:
        return "missing_or_invalid"
    if a & b:
        return "agreement" if b <= a else "mixed_agreement_and_disagreement"
    return "disagreement_all_observed"

assert identifier_comparison(["0000123"], ["123"], positive_digits) == "agreement"
assert identifier_comparison([pd.NA], ["123"], positive_digits) == "missing_or_invalid"
assert identifier_comparison(["000"], ["0"], positive_digits) == "missing_or_invalid"

pair_rows = []
non_distinctive_tokens = {"the", "and", "of", "group", "holdings", "holding", "financial", "energy",
                        "resources", "bank", "systems", "international", "technologies", "services", "co",
                        "petroleum", "oil", "mining", "exploration"}
for candidate in additional_local_candidates.itertuples(index=False):
    gvkey, rcid = candidate.gvkey, candidate.rcid
    source = audit_source.loc[audit_source.gvkey.eq(gvkey)]
    reference = audit_reference.loc[audit_reference.rcid_key.eq(rcid)]
    annual = source.loc[source.source.eq("compustat")]
    crsp = source.loc[source.source.eq("crsp")]
    evidence = local_candidate_evidence.loc[local_candidate_evidence.gvkey.eq(gvkey) & local_candidate_evidence.rcid.eq(rcid)]
    research_names, candidate_names = original_values(source.company_name), original_values(reference.company)
    source_names = {name_key(v) for v in research_names} - {""}
    ref_names = {name_key(v) for v in candidate_names} - {""}
    name_result = identifier_comparison(research_names, candidate_names, name_key)
    raw_name_agreement = bool(set(research_names) & set(candidate_names))
    shared_tokens = (set(" ".join(source_names).split()) & set(" ".join(ref_names).split())) - non_distinctive_tokens
    cik_result = identifier_comparison(source.cik, reference.cik, positive_digits)
    gvkey_result = identifier_comparison([gvkey], reference.gvkey, positive_digits)
    cusip_result, equivalent_pairs, unmatched_ref_cusips = compare_cusips(source.cusip, reference.cusip)
    source_cusip8 = {cusip_audit_key(v)[0][:8] for v in source.cusip} - {""}
    name_variation = len(source_names) > 1
    cusip_variation = len(source_cusip8) > 1
    exact_or_prefix = cusip_result in {"exact_full_cusip_agreement", "cusip8_format_agreement", "cusip8_agreement_9_vs_9"}
    hard_conflict = cik_result == "disagreement_all_observed" or gvkey_result == "disagreement_all_observed"
    mixed = "mixed_agreement_and_disagreement" in [cik_result, gvkey_result, cusip_result]
    lexical_history_cue = bool(shared_tokens) and (hard_conflict or cusip_result == "disagreement_all_valid_observed")
    corroborated = (cik_result == "agreement" or gvkey_result == "agreement") and (name_result == "agreement" or exact_or_prefix)
    credible = corroborated and not hard_conflict and not mixed and cusip_result != "disagreement_all_valid_observed"
    n_rcids = int(new_counts.loc[gvkey])
    if lexical_history_cue or mixed:
        category = "needs_historical_review"
    elif cusip_result == "disagreement_all_valid_observed" and (hard_conflict or name_result == "disagreement_all_observed"):
        category = "likely_false_positive"
    elif n_rcids > 1:
        category = "ambiguous_multiple_rcids"
    elif credible:
        category = "strong_candidate"
    elif exact_or_prefix or cik_result == "agreement" or gvkey_result == "agreement" or name_result == "agreement":
        category = "needs_historical_review"
    else:
        category = "insufficient_evidence"
    explanations = []
    for label, status in [("GVKEY", gvkey_result), ("CIK", cik_result), ("CUSIP", cusip_result), ("name", name_result)]:
        explanations.append(label + ": " + status)
    if cusip_variation: explanations.append("multiple observed security identifiers; cause unestablished")
    if lexical_history_cue: explanations.append("shared name tokens " + ", ".join(sorted(shared_tokens)) + "; relationship unproven")
    if n_rcids > 1: explanations.append("multiple ticker candidates; no RCID selected")
    parents = original_values(reference.ultimate_parent_rcid)
    parent_scope = bool(set(parents) - {rcid})
    other_owners = tuple(sorted(baseline_rcid_owners.get(rcid, set()) - {gvkey}))
    if parent_scope: explanations.append("different parent RCID in undated Revelio snapshot")
    if other_owners: explanations.append("RCID also a baseline candidate for GVKEY " + ", ".join(other_owners))
    # Retain locally available identity evidence for the candidate's own reported GVKEY.
    candidate_history = compustat_source.loc[compustat_source.gvkey.map(positive_digits).isin(
        {positive_digits(v) for v in reference.gvkey} - {""})]
    dates = sorted({d for ds in crsp.observed_dates for d in ds})
    priority = "P1" if category in {"needs_historical_review", "insufficient_evidence", "ambiguous_multiple_rcids"} or n_rcids > 1 else "P2" if category == "strong_candidate" else "P3"
    pair_rows.append({
        "gvkey": gvkey, "research_company_names": research_names, "rcid": rcid,
        "revelio_company_names": candidate_names, "matching_tickers": tuple(sorted(evidence.comparison_key.unique())),
        "ticker_sources": tuple(sorted({v["source"] for v in evidence.source_evidence})),
        "compustat_tickers": original_values(annual.ticker), "crsp_tickers": original_values(crsp.ticker),
        "research_cik_raw": original_values(source.cik), "revelio_cik_raw": original_values(reference.cik),
        "cik_result": cik_result, "compustat_cusip_raw": original_values(annual.cusip),
        "crsp_cusip_raw": original_values(crsp.cusip), "revelio_cusip_raw": original_values(reference.cusip),
        "cusip_result": cusip_result, "equivalent_cusip_pairs": equivalent_pairs,
        "source_cusip8_variation": cusip_variation,
        "dated_cusip_transition_count": int(cusip_historical_transitions.gvkey.eq(gvkey).sum()),
        "cusip_invalid_source_values": tuple(sorted({text_key(v) for v in source.cusip if cusip_audit_key(v)[1].startswith("invalid")})),
        "cusip_invalid_revelio_values": tuple(sorted({text_key(v) for v in reference.cusip if cusip_audit_key(v)[1].startswith("invalid")})),
        "raw_name_agreement": raw_name_agreement, "research_names_normalized": tuple(sorted(source_names)),
        "revelio_names_normalized": tuple(sorted(ref_names)), "name_result": name_result,
        "research_name_variation": name_variation, "shared_name_tokens_review_only": tuple(sorted(shared_tokens)),
        "revelio_gvkey_raw": original_values(reference.gvkey), "gvkey_result": gvkey_result,
        "annual_reporting_dates": original_values(annual.datadate),
        "crsp_observed_date_bounds": (dates[0], dates[-1]) if dates else (), "crsp_observed_date_count": len(dates),
        "candidate_own_compustat_identity": tuple(candidate_history[["gvkey", "conm", "cik", "cusip", "tic"]].drop_duplicates().itertuples(index=False,name=None)),
        "candidate_own_reporting_dates": original_values(candidate_history.datadate),
        "parent_rcids": parents, "parent_names": original_values(reference.ultimate_parent_company_name),
        "child_rcids": original_values(reference.child_rcid), "child_names": original_values(reference.child_company),
        "parent_scope_review": parent_scope, "baseline_other_gvkeys_for_rcid": other_owners,
        "possible_source_ticker_reuse": bool(candidate.ticker_reuse_or_conflict),
        "revelio_ticker_identity_collision": category == "likely_false_positive",
        "historical_relationship_status": "unestablished: no dated transaction/ownership evidence",
        "rcid_candidate_count": n_rcids, "original_cusip_conflict": bool(candidate.cusip_conflict),
        "credible_requiring_final_review": bool(credible), "conflict_explanations": "; ".join(explanations),
        "review_priority": priority, "provisional_review_category": category,
        "source_evidence_ids": tuple(evidence.evidence_id), "revelio_source_rows": tuple(reference.revelio_source_row),
    })
candidate_quality_audit = pd.DataFrame(pair_rows).sort_values(["review_priority", "gvkey", "rcid"]).reset_index(drop=True)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
    display(candidate_quality_audit)

#### 2.3.5 Summary, review queue, and reconciliation

Candidate categories partition the 73 records. Distinct-GVKEY counts **within** categories
can overlap; do not sum them as firm coverage. A second table assigns each of the 43 firms
one deterministic triage category in priority order: credible candidate, historical review,
insufficient/ambiguous evidence, then all candidates likely false positives. This is a firm
review summary, not a preferred RCID. The full P1 queue identifies exact GVKEY/RCID pairs.

In [ ]:
categories = ["strong_candidate", "needs_historical_review", "ambiguous_multiple_rcids", "likely_false_positive", "insufficient_evidence"]
category_summary = candidate_quality_audit.groupby("provisional_review_category").agg(
    candidate_records=("rcid", "size"), distinct_gvkeys=("gvkey", "nunique")
).reindex(categories, fill_value=0)
category_summary["gvkeys_within_category"] = [tuple(sorted(candidate_quality_audit.loc[
    candidate_quality_audit.provisional_review_category.eq(cat), "gvkey"].unique())) for cat in categories]
display(category_summary)
firm_triage_rows = []
for gvkey, group in candidate_quality_audit.groupby("gvkey"):
    cats = set(group.provisional_review_category)
    triage = ("credible_candidate_final_review" if group.credible_requiring_final_review.any() else
              "historical_review" if "needs_historical_review" in cats else
              "insufficient_or_ambiguous" if cats & {"insufficient_evidence", "ambiguous_multiple_rcids"} else
              "all_candidates_likely_false_positive")
    firm_triage_rows.append({"gvkey": gvkey, "triage": triage, "rcid_candidates": len(group),
                            "unresolved_multiple_rcids": len(group) > 1,
                            "credible_candidates": int(group.credible_requiring_final_review.sum())})
firm_quality_triage = pd.DataFrame(firm_triage_rows)
display(firm_quality_triage.groupby("triage").size().rename("mutually_exclusive_firms").to_frame())
cusip_flag_reconciliation = candidate_quality_audit.groupby("cusip_result").agg(
    original_flagged_records=("original_cusip_conflict", "sum"), distinct_gvkeys=("gvkey", "nunique"))
display(cusip_flag_reconciliation)
manual_first_review = candidate_quality_audit.loc[candidate_quality_audit.review_priority.eq("P1"), [
    "gvkey", "research_company_names", "rcid", "revelio_company_names", "provisional_review_category", "conflict_explanations"]]
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
    display(manual_first_review)

assert len(candidate_quality_audit) == 73 and candidate_quality_audit.gvkey.nunique() == 43
assert not candidate_quality_audit.duplicated(["gvkey", "rcid"]).any()
assert set(candidate_quality_audit[["gvkey", "rcid"]].itertuples(index=False,name=None)) == set(
    additional_local_candidates[["gvkey", "rcid"]].itertuples(index=False,name=None))
assert category_summary.candidate_records.sum() == 73
assert len(firm_quality_triage) == 43 and firm_quality_triage.gvkey.is_unique
assert firm_quality_triage.unresolved_multiple_rcids.sum() == 16
pd.testing.assert_frame_equal(additional_local_candidates, audit_candidates_before)
pd.testing.assert_frame_equal(remaining_unmatched_review, audit_unmatched_before)
pd.testing.assert_frame_equal(local_firm_review_index, audit_firms_before)
pd.testing.assert_frame_equal(crosswalk_v1, pd.read_parquet(input_paths["crosswalk_v1"]))
assert frozenset(all_candidate_triples) == audit_triples_before
assert saved_candidates == reconstructed_candidates and len(saved_candidates) == 1638
assert len(remaining_local) == 44 and len(local_firm_review_index) == 652
assert crosswalk_v1.rcid.isna().all() and local_firm_review_index.accepted_rcid.isna().all()
print("PASS: audit-only; all 652 firms, 1,638 original triples, 73 new pairs, 608 candidate-covered firms and 44 unmatched firms preserved.")
print("Accepted RCIDs: 0. Part 2.4 and crosswalk export not implemented.")

#### 2.3.6 Findings and required logic correction

The full-string flag from Part 2.2 is retained for reproducibility. The corrected
CUSIP diagnostic here first validates identifiers, then distinguishes exact agreement,
8/9 format equivalence, missing/invalid evidence, and disagreement with all observed
security identifiers. This correction is needed in future conflict reporting; it does
not authorize CUSIP8 matching, modify the baseline, or accept an identity.

Historical observations establish which identifiers appeared in these files on observed
dates. They do not explain corporate events. A differently named foreign or domestic
company sharing a ticker is an observed collision, not evidence of a subsidiary or
successor. Source-side ticker-reuse flags alone miss cross-exchange collisions because
the local CRSP universe is limited; exchange and effective-date fields would be needed
to disambiguate those cases reliably. No outside facts were used.

In [ ]:
format_count = int(candidate_quality_audit.cusip_result.eq("cusip8_format_agreement").sum())
disagree_count = int(candidate_quality_audit.cusip_result.eq("disagreement_all_valid_observed").sum())
credible_firms = int(firm_quality_triage.credible_candidates.gt(0).sum())
unresolved_multiple = int(firm_quality_triage.unresolved_multiple_rcids.sum())
lines = ["**Candidate quality findings (diagnostic only)**", ""]
for cat, row in category_summary.iterrows():
    lines.append(f"- `{cat}`: {row.candidate_records} candidate records, {row.distinct_gvkeys} distinct GVKEYs within category.")
lines += [
    f"- Of the 73 original CUSIP flags, **{format_count} are 8/9 format-only agreements** and **{disagree_count} disagree with all valid observed CUSIPs**. "
    "The residual missing/invalid/mixed statuses are reported separately above. Observed security disagreement is not proof of a corporate identity contradiction by itself.",
    f"- **{credible_firms}/43 firms** have a candidate meeting the conservative corroboration rule. **{unresolved_multiple} multiple-RCID groups** remain unadjudicated; no candidate has been removed.",
    f"- {candidate_quality_audit.loc[candidate_quality_audit.source_cusip8_variation, 'gvkey'].nunique()} audited firms show more than one observed CUSIP8. "
    f"There are {len(cusip_historical_transitions)} observed same-PERMNO dated transitions. Variation does not establish acquisitions or renames.",
    "- Review the complete P1 queue first: historical/mixed identity cues, missing source evidence, and all multiple-RCID groups. "
    "Remaining P3 cases combine security disagreement with another identity or name contradiction and are likely ticker false positives.",
]
fox = candidate_quality_audit.loc[candidate_quality_audit.gvkey.eq("012886") & candidate_quality_audit.rcid.eq("430242")]
if len(fox):
    lines.append("- Priority historical case: **GVKEY 012886 / RCID 430242**, Twenty-First Century Fox versus Fox Corp. "
                 "Local evidence reports different GVKEY, CIK and CUSIP; Fox Corp also has separate Compustat records under 034636. "
                 "The shared name token/ticker is a review cue, not proof of succession, an acquisition, or an interchangeable RCID. "
                 "Dated corporate-action or legal-entity evidence is absent, so this relationship remains unresolved.")
for row in candidate_quality_audit.loc[candidate_quality_audit.provisional_review_category.isin(
        ["insufficient_evidence", "ambiguous_multiple_rcids"])].itertuples():
    lines.append(f"- Source-gap review: GVKEY **{row.gvkey} / RCID {row.rcid}**, {', '.join(row.revelio_company_names)}. "
                 "No observed research-company name/CIK corroborates the candidate; CUSIP disagreement remains. "
                 "Obtain independent dated identity evidence before any decision.")
lines += [
    "- The original 565 firms retain their candidates; their quality is outside this audit. Raw candidate coverage remains 608/652, "
    "with 44 unmatched. That coverage must not be represented as verified entity coverage. No RCIDs were accepted.",
    "- Complete raw identifiers, normalized comparisons, dated evidence, parent/child snapshot fields, conflict explanations and exact "
    "GVKEY/RCID review priorities are retained above. Fuller company-reference and dated entity-history evidence may be needed, "
    "but no downloads or additional matching were performed.",
]
display(Markdown("\n".join(lines)))